# Explainable Multi-Stage Deep Learning Framework for Brain Tumor MRI Analysis

**Pipeline:** Raw MRI → U-Net Segmentation → Tumor ROI Cropping → ResNet50 Classification → Grad-CAM + VLM Report

This notebook implements the three-stage architecture described in our COMP4471 project:

| Stage | Component | Description |
|-------|-----------|-------------|
| 0 | **Preprocessing** | Data loading, EDA, stratified split, segmentation dataset preparation |
| 1 | **Segmentation** | Standard 2D U-Net – Dice ≥ 0.7, fast training on A100 |
| 2 | **Classification** | ResNet50 (ImageNet transfer) – target >95% accuracy/F1 |
| 3 | **Explainability** | Grad-CAM heatmaps + CLIP-based VLM medical report generation |

---

## Datasets
- **Kaggle Brain Tumor MRI Dataset** (~7,000 images, 4 classes) – [link](https://www.kaggle.com/datasets/sartajbhuvaji/brain-tumor-classification-mri)
- **Kaggle Brain Tumor Segmentation Dataset** (3,064 paired images + masks) – [link](https://www.kaggle.com/datasets/nikhilroxtomar/brain-tumor-segmentation)

## Reproducibility
All random seeds are fixed. Run cells sequentially from top to bottom.

---
## 0.1 Environment Setup & Installation

In [ ]:
# ============================================================
# Cell 0.0 – Google Drive mount & dependency install
# ============================================================
import os, sys, subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

_NEED = False
for _pkg in ['medpy', 'cv2', 'skimage']:
    try:
        __import__(_pkg)
    except (ImportError, ValueError):
        _NEED = True
        break

if _NEED:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'medpy', 'opencv-python-headless', 'scikit-image',
        'nibabel', 'albumentations', 'grad-cam', 'captum',
        'open_clip_torch', 'omegaconf', 'kaggle', 'transformers'
    ])
    print('Packages installed (including transformers).')
else:
    print('All dependencies already installed.')

In [ ]:
# ============================================================
# Cell 0.1 – PROJECT_ROOT (Colab / Local compatible version)
# ============================================================

# 1. Colab uses Google Drive paths first
if IN_COLAB:
    _CANDIDATES = [
        Path("/content/drive/MyDrive/comp4471Project"),
    ]
else:
    # Local: use current working directory or parent
    _CANDIDATES = [Path.cwd().parent, Path.cwd()]

PROJECT_ROOT = None
for _p in _CANDIDATES:
    if (_p / "src" / "utils.py").exists():
        PROJECT_ROOT = _p
        break

# 2. Final fallback using environment variable or relative path
if PROJECT_ROOT is None:
    env_root = os.getenv("PROJECT_ROOT")
    if env_root:
        PROJECT_ROOT = Path(env_root)
    else:
        # Last resort: walk up from current file or cwd
        PROJECT_ROOT = Path(__file__).resolve().parent.parent if '__file__' in globals() else Path.cwd().parent

assert PROJECT_ROOT is not None and (PROJECT_ROOT / "src" / "utils.py").exists(), \
    "Could not find src/utils.py! Please check project structure or set PROJECT_ROOT environment variable."

print(f" PROJECT_ROOT = {PROJECT_ROOT}")
print(f"   Has src/     : {(PROJECT_ROOT/'src').exists()}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 3. Force Python cache refresh only on Colab
if IN_COLAB:
    try:
        import ctypes
        libc = ctypes.CDLL("libc.so.6")
        for _pyf in (PROJECT_ROOT / "src").glob("*.py"):
            fd = os.open(str(_pyf), os.O_RDONLY)
            libc.fdatasync(fd)
            os.close(fd)
    except:
        pass  # non-critical

---
## 0.2 Imports & Reproducibility

In [ ]:
# ============================================================
# Cell 0.2 – Imports & seed
# ============================================================
import warnings
warnings.filterwarnings("ignore")

import os, sys
from pathlib import Path

if "PROJECT_ROOT" not in dir():
    _candidates = [Path.cwd().parent, Path.cwd()]
    for _p in _candidates:
        if (_p / "src" / "utils.py").exists():
            PROJECT_ROOT = _p
            break
    else:
        raise RuntimeError("Cannot find src/utils.py – run Cell 0.1 first or set PROJECT_ROOT")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from PIL import Image

# Force-reload ALL src modules（重要修正）
import src.utils, src.preprocessing, src.segmentation, src.classification
import src.explainability, src.visualization

for _m in [src.utils, src.preprocessing, src.segmentation,
           src.classification, src.explainability, src.visualization]:
    importlib.reload(_m)

from src.utils import seed_everything, get_device, load_config, EarlyStopping, AverageMeter
from src.preprocessing import (
    discover_images, stratified_split, stratified_split_pairs,
    CLASS_NAMES, CLASS_TO_IDX,
    crop_tumor_region, generate_pseudo_mask, generate_pseudo_masks_for_split,
    discover_image_mask_pairs, SegmentationDataset,
)
from src.segmentation import (
    UNet, ResUNet, DiceBCELoss, train_unet, predict_mask, predict_batch,
    compute_segmentation_metrics, evaluate_segmentation,
)
from src.classification import (
    BrainTumorDataset, build_resnet50, compute_class_weights,
    get_train_transforms, get_val_transforms,
    train_one_epoch, evaluate, train_classifier,
    grid_search_hyperparams,
)
from src.explainability import (
    generate_gradcam, overlay_gradcam,
    generate_template_report, CLIPReportGenerator,
    compute_stage_consistency_iou, generate_uncertainty_map,
)
from src.visualization import (
    plot_segmentation_overlay, plot_confusion_matrix,
    plot_training_curves, plot_qualitative_panel,
    plot_class_distribution,
)

SEED = 42
seed_everything(SEED)
DEVICE = get_device()
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

cfg = load_config(str(PROJECT_ROOT / "configs" / "pipeline_config.yaml"))
print(f"Config loaded. Classes: {cfg['dataset']['classes']}")

---
## 0.3 Configuration & Paths

In [ ]:
# ============================================================
# Cell 0.3 – Project paths (PROJECT_ROOT set in Cell 0.1)
# ============================================================
RAW_DATA_DIR   = PROJECT_ROOT / cfg["paths"]["raw_data_dir"]
PROCESSED_DIR  = PROJECT_ROOT / cfg["paths"]["processed_data_dir"]
SEG_MODEL_DIR  = PROJECT_ROOT / cfg["paths"]["segmentation_model_dir"]
CLS_MODEL_DIR  = PROJECT_ROOT / cfg["paths"]["classification_model_dir"]
VLM_MODEL_DIR  = PROJECT_ROOT / cfg["paths"]["vlm_model_dir"]
OUTPUT_DIR     = PROJECT_ROOT / cfg["paths"]["output_dir"]

for d in [PROCESSED_DIR, SEG_MODEL_DIR, CLS_MODEL_DIR, VLM_MODEL_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data dir: {RAW_DATA_DIR}")

---
# Stage 0 – Data Preparation & Exploratory Data Analysis

Load the Kaggle Brain Tumor MRI Dataset, inspect class distribution, and create
stratified 80/10/10 train/val/test splits.

In [ ]:
# ============================================================
# Cell 0.4 – Download the new single combined dataset
# ============================================================
# Kaggle credentials handling

if IN_COLAB:
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
        print("Kaggle credentials loaded from Colab Secrets.")
    except:
        print("Warning: Could not load Kaggle credentials from Colab Secrets.")


if RAW_DATA_DIR.exists() and any(RAW_DATA_DIR.iterdir()):
    print(f'Dataset already exists at {RAW_DATA_DIR} – skipping download.')
else:
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(['kaggle', 'datasets', 'download', '-d',
                    'indk214/brain-tumor-dataset-segmentation-and-classification',
                    '-p', str(RAW_DATA_DIR), '--unzip'], check=True)
    print(f'New combined dataset downloaded to {RAW_DATA_DIR}')

print('Dataset download completed.')

In [ ]:
# ============================================================
# Cell 0.5 – Discover images & class distribution
# ============================================================
# Uses discover_images() from src/preprocessing.py which handles
# all Kaggle folder-name variants (glioma_tumor, meningioma_tumor, etc.)

samples = discover_images(str(RAW_DATA_DIR))
print(f"Total images found: {len(samples)}")
print(f"CLASS_NAMES: {CLASS_NAMES}")
print(f"CLASS_TO_IDX: {CLASS_TO_IDX}")

assert len(samples) > 0, (
    f"No images found in {RAW_DATA_DIR}. Check that Training/ and Testing/ "
    f"subdirectories exist with class folders."
)

labels = [s[1] for s in samples]
label_counts = Counter(labels)
print("\nClass distribution:")
for cls_idx, count in sorted(label_counts.items()):
    print(f"  {CLASS_NAMES[cls_idx]:>12s}: {count:>5d}")

plot_class_distribution(
    labels,
    class_names=CLASS_NAMES,
    title="Kaggle Brain Tumor MRI - Class Distribution",
    save_path=str(OUTPUT_DIR / "figures" / "class_distribution.png"),
)

In [ ]:
# ============================================================
# Cell 0.6 – Stratified train/val/test split (80/10/10)
# ============================================================
train_samples, val_samples, test_samples = stratified_split(
    samples, ratios=(0.8, 0.1, 0.1), seed=SEED
)

print(f"Train: {len(train_samples)}  |  Val: {len(val_samples)}  |  Test: {len(test_samples)}")

for split_name, split_data in [("Train", train_samples), ("Val", val_samples), ("Test", test_samples)]:
    split_labels = [s[1] for s in split_data]
    counts = Counter(split_labels)
    print(f"\n{split_name} split:")
    for cls_idx in sorted(counts):
        print(f"  {CLASS_NAMES[cls_idx]:>12s}: {counts[cls_idx]:>5d}")

In [ ]:
# ============================================================
# Cell 0.7 – Visualise sample images from each class
# ============================================================
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("Sample MRI Images per Class", fontsize=16, fontweight="bold")

for cls_idx, cls_name in enumerate(CLASS_NAMES):
    cls_samples = [s for s in train_samples if s[1] == cls_idx]
    for row in range(2):
        if row < len(cls_samples):
            img = Image.open(cls_samples[row][0]).convert("L")
            axes[row, cls_idx].imshow(np.array(img), cmap="gray")
        axes[row, cls_idx].axis("off")
        if row == 0:
            axes[row, cls_idx].set_title(cls_name, fontsize=13)

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "figures" / "sample_images.png"), dpi=150, bbox_inches="tight")
plt.show()

---
# Stage 1 – U-Net Training with REAL Masks (Albumentations + True Annotations)

This stage trains a standard 2D U-Net on **real paired image-mask data** from the
Kaggle Brain Tumor Segmentation dataset (3,064 paired MRI slices + binary masks).

| Component | Detail |
|-----------|--------|
| **Data** | `discover_image_mask_pairs()` → real annotations (no Otsu pseudo-masks) |
| **Split** | Stratified 80/10/10 by tumor class via `stratified_split_pairs()` |
| **Augmentation** | Strong Albumentations pipeline (flips, elastic, grid distortion, CLAHE, noise, etc.) |
| **Architecture** | 4-level encoder-decoder, double-conv blocks, skip connections, 64 base filters |
| **Loss** | Dice + BCE combined loss |
| **Resolution** | 320 × 320 |

**Target metrics:** Dice ≥ 0.88, Hausdorff-95 ≤ 5.0

In [ ]:
# ============================================================
# Stage 1 – Full pipeline: real masks, stratified split, train, evaluate
# ============================================================
import time as _time
import pickle
from collections import Counter

# ── 1. Discover real image-mask pairs ────────────────────────
# Points to the segmentation dataset with class subfolders
# containing interleaved *_mask.png files (Layout A).
SEG_DATA_DIR = str(RAW_DATA_DIR / "Segmentation")
pairs = discover_image_mask_pairs(SEG_DATA_DIR)

print(f"Real image-mask pairs found: {len(pairs)}")
print(f"  Example: {Path(pairs[0][0]).name}  <->  {Path(pairs[0][1]).name}")

In [ ]:

# ── 2. Stratified 80/10/10 split ────────────────────────────
# Stratifies by parent folder name (= tumor class) so each
# split has proportional representation of Glioma / Meningioma /
# Pituitary tumor.
seg_train, seg_val, seg_test = stratified_split_pairs(
    pairs, ratios=(0.8, 0.1, 0.1), seed=SEED,
)

print(f"\nStratified split:")
print(f"  Train: {len(seg_train)}  |  Val: {len(seg_val)}  |  Test: {len(seg_test)}")

cls_dist = Counter(Path(img).parent.name for img, _ in seg_train)
print(f"  Train class distribution: {dict(cls_dist)}")

# Persist splits for reproducibility
for name, data in [("seg_train", seg_train), ("seg_val", seg_val), ("seg_test", seg_test)]:
    with open(PROCESSED_DIR / f"{name}_pairs.pkl", "wb") as f:
        pickle.dump(data, f)

In [ ]:
# ── 3. Build SegmentationDataset instances ───────────────────
# img_size=320 gives the model more spatial detail than 256.
# augment=True activates the full Albumentations pipeline:
#   HorizontalFlip, VerticalFlip, RandomRotate90, ShiftScaleRotate,
#   ElasticTransform, GridDistortion, RandomBrightnessContrast,
#   CLAHE, GaussNoise, Normalize(mean=0.5, std=0.5), ToTensorV2.
# Val/test use only Resize + Normalize + ToTensorV2.
SEG_IMG_SIZE = 320

seg_train_ds = SegmentationDataset(seg_train, img_size=SEG_IMG_SIZE, augment=True)
seg_val_ds   = SegmentationDataset(seg_val,   img_size=SEG_IMG_SIZE, augment=False)
seg_test_ds  = SegmentationDataset(seg_test,  img_size=SEG_IMG_SIZE, augment=False)


In [ ]:


# ── 4. DataLoaders ──────────────────────────────────────────
SEG_BATCH_SIZE = 16

seg_train_loader = DataLoader(
    seg_train_ds, batch_size=SEG_BATCH_SIZE, shuffle=True,
    num_workers=4, pin_memory=True,
)
seg_val_loader = DataLoader(
    seg_val_ds, batch_size=SEG_BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True,
)
seg_test_loader = DataLoader(
    seg_test_ds, batch_size=SEG_BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True,
)

print(f"\nDataLoaders ready:")
print(f"  Train: {len(seg_train_loader)} batches  (bs={SEG_BATCH_SIZE})")
print(f"  Val:   {len(seg_val_loader)} batches")
print(f"  Test:  {len(seg_test_loader)} batches")

# Quick sanity check on one batch
imgs, masks = next(iter(seg_train_loader))
print(f"  Batch shapes: images={imgs.shape}, masks={masks.shape}")
print(f"  Image range:  [{imgs.min():.2f}, {imgs.max():.2f}]")
print(f"  Mask  unique: {masks.unique().tolist()}")

In [ ]:
# ── 5. Initialize & train ResUNet ───────────────────────────
seg_model = ResUNet(in_channels=1, out_channels=1, base_filters=64).to(DEVICE)
total_params = sum(p.numel() for p in seg_model.parameters())
print(f"\nResUNet parameters: {total_params:,}")

SEG_SAVE_PATH = str(SEG_MODEL_DIR / "best_resunet.pth")

if Path(SEG_SAVE_PATH).exists():
    print(f"Trained model found at {SEG_SAVE_PATH} - loading weights.")
    seg_model.load_state_dict(torch.load(SEG_SAVE_PATH, map_location=DEVICE))
    seg_model = seg_model.to(DEVICE)
else:
    _t0 = _time.time()
    seg_history = train_unet(
        model=seg_model,
        train_loader=seg_train_loader,
        val_loader=seg_val_loader,
        device=DEVICE,
        epochs=130,
        lr=3e-4,
        patience=12,
        save_path=SEG_SAVE_PATH,
    )
    _elapsed = _time.time() - _t0
    _m, _s = divmod(int(_elapsed), 60)
    print(f"\nTraining complete in {_m}m {_s:02d}s.")

    # Reload best checkpoint
    seg_model.load_state_dict(torch.load(SEG_SAVE_PATH, map_location=DEVICE))
    seg_model = seg_model.to(DEVICE)

# Keep an explicit ResUNet handle for downstream cells.
resunet_model = seg_model

In [ ]:
# ── 6. Test-set evaluation ──────────────────────────────────
seg_eval_model = globals().get("seg_model", globals().get("resunet_model", None))
if seg_eval_model is None or not hasattr(seg_eval_model, "eval"):
    raise RuntimeError("ResUNet model is not available. Run the segmentation train/load cell first.")

seg_avg, seg_std = evaluate_segmentation(seg_eval_model, seg_test_loader, DEVICE)

print(f"\n{'=' * 55}")
print(f"  TEST SET RESULTS  (n={len(seg_test)})")
print(f"{'=' * 55}")
print(f"  Dice:          {seg_avg['dice']:.4f}")
print(f"  Hausdorff-95:  {seg_avg['hausdorff95']:.2f}")
print(f"  Sensitivity:   {seg_avg['sensitivity']:.4f}")
print(f"  Specificity:   {seg_avg['specificity']:.4f}")
print(f"{'=' * 55}")

In [ ]:
# ============================================================
# Cell - Load ResUNet Checkpoint
# ============================================================

import torch
from pathlib import Path

full_checkpoint_path = SEG_MODEL_DIR / "resunet_full_checkpoint.pth"
weights_path = SEG_MODEL_DIR / "best_resunet.pth"

seg_model = ResUNet(in_channels=1, out_channels=1).to(DEVICE)

if full_checkpoint_path.exists():
    checkpoint = torch.load(full_checkpoint_path, map_location=DEVICE)

    # Support both full checkpoint dicts and raw state_dict files.
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        seg_model.load_state_dict(checkpoint["model_state_dict"])
        seg_history = checkpoint.get("history", {})
    else:
        seg_model.load_state_dict(checkpoint)
        seg_history = {}

    print("ResUNet model loaded successfully!")
    if isinstance(seg_history, dict) and "train_loss" in seg_history:
        print(f"Number of trained epochs: {len(seg_history['train_loss'])}")
    if isinstance(seg_history, dict) and seg_history.get("val_dice"):
        print(f"Current best validation Dice: {max(seg_history['val_dice']):.4f}")
elif weights_path.exists():
    seg_model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
    print(f"ResUNet weights loaded successfully from {weights_path}.")
else:
    print("ResUNet checkpoint file does not exist. Please train/save ResUNet first.")

# Keep an explicit ResUNet handle for downstream cells.
resunet_model = seg_model

In [ ]:
# ============================================================
# Cell 1.x - Save segmentation model + training history
# ============================================================

import torch
from pathlib import Path

# Ensure model storage directory exists
SEG_MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_to_save = globals().get("seg_model", globals().get("resunet_model", None))
if model_to_save is None or not hasattr(model_to_save, "state_dict"):
    raise RuntimeError("No valid ResUNet model found. Run the segmentation load/train cell first.")

model_name = type(model_to_save).__name__
checkpoint_filename = "resunet_full_checkpoint.pth" if "resunet" in model_name.lower() else "unet_full_checkpoint.pth"
checkpoint_path = SEG_MODEL_DIR / checkpoint_filename

history_payload = globals().get("seg_history", None)
if not isinstance(history_payload, dict):
    history_payload = {}
    print("seg_history not found in this session; saving checkpoint with empty history.")

torch.save(
    {
        "model_state_dict": model_to_save.state_dict(),
        "history": history_payload,
    },
    checkpoint_path,
)

epochs = len(history_payload.get("train_loss", []))
print(f"{model_name} checkpoint saved successfully!")
print(f"Path: {checkpoint_path}")
print(f"Tracked epochs in history: {epochs}")
print(f"Number of model parameters: {sum(p.numel() for p in model_to_save.parameters()):,}")

In [ ]:
# ============================================================
# Cell 1.3 – Plot segmentation training curves
# ============================================================
required_keys = ('train_loss', 'val_loss', 'val_dice')

if isinstance(globals().get('seg_history'), dict):
    missing = [k for k in required_keys if k not in seg_history or len(seg_history.get(k, [])) == 0]

    if not missing:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        n_epochs = min(len(seg_history['train_loss']), len(seg_history['val_loss']), len(seg_history['val_dice']))
        epochs_range = range(1, n_epochs + 1)

        axes[0].plot(epochs_range, seg_history['train_loss'][:n_epochs], label='Train Loss')
        axes[0].plot(epochs_range, seg_history['val_loss'][:n_epochs], label='Val Loss')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].set_title('ResUNet Training & Validation Loss')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        axes[1].plot(epochs_range, seg_history['val_dice'][:n_epochs], label='Val Dice', color='green')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Dice Score')
        axes[1].set_title('ResUNet Validation Dice')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        (OUTPUT_DIR / 'figures').mkdir(parents=True, exist_ok=True)
        plt.savefig(str(OUTPUT_DIR / 'figures' / 'resunet_training_curves.png'), dpi=150, bbox_inches='tight')
        plt.show()
    else:
        print('Training history is incomplete; cannot plot segmentation curves.')
        print(f'Missing keys: {missing}')
        print(f'Available keys: {sorted(seg_history.keys())}')
        print('Tip: load the full checkpoint or run segmentation training to populate history.')
else:
    print('Training history not available (model was loaded from checkpoint).')

## 1.4 Segmentation Evaluation

In [ ]:
# ============================================================
# Cell 1.5 – Visualise segmentation results
# ============================================================
seg_vis_model = globals().get("seg_model", globals().get("resunet_model", None))
if seg_vis_model is None or not hasattr(seg_vis_model, "eval"):
    raise RuntimeError("ResUNet model is not available. Run the segmentation train/load cell first.")

seg_vis_model.eval()
(OUTPUT_DIR / 'figures').mkdir(parents=True, exist_ok=True)

num_vis = min(4, len(seg_test))
fig, axes = plt.subplots(num_vis, 3, figsize=(15, 5 * num_vis))
if num_vis == 1:
    axes = axes[np.newaxis, :]

for i in range(num_vis):
    img_path, mask_path = seg_test[i]
    img = np.array(Image.open(img_path).convert('L'))
    gt  = np.array(Image.open(mask_path).convert('L'))
    gt  = (gt > 127).astype(np.uint8)

    pred = predict_mask(seg_vis_model, img, DEVICE)

    axes[i, 0].imshow(img, cmap='gray')
    axes[i, 0].set_title('Original MRI')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(img, cmap='gray')
    axes[i, 1].imshow(gt, cmap='Reds', alpha=0.4)
    axes[i, 1].set_title('Ground-Truth Mask')
    axes[i, 1].axis('off')

    axes[i, 2].imshow(img, cmap='gray')
    axes[i, 2].imshow(pred, cmap='Blues', alpha=0.4)
    axes[i, 2].set_title('Predicted Mask')
    axes[i, 2].axis('off')

plt.suptitle('ResUNet Segmentation Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'figures' / 'seg_overlay_resunet.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

metrics_dir = OUTPUT_DIR / "metrics" if "OUTPUT_DIR" in globals() else Path("outputs/metrics")
comparison_csv = metrics_dir / "unet_resunet_attention_comparison.csv"


def _load_metrics(path: Path, label: str):
    if not path.exists():
        print(f"Warning: {label} metrics file not found -> {path}")
        return {}
    try:
        return json.loads(path.read_text())
    except Exception as e:
        print(f"Warning: failed to parse {label} metrics from {path}: {e}")
        return {}


def _metric(payload: dict, keys, default=np.nan):
    for key in keys:
        if key in payload and payload[key] is not None:
            try:
                return float(payload[key])
            except Exception:
                continue
    return float(default)


def _recover_unet_metrics_if_missing(payload: dict):
    if payload:
        return payload

    if "seg_test_loader" not in globals():
        print("Warning: cannot recover U-Net metrics because seg_test_loader is unavailable in this session.")
        return {}

    try:
        import torch
        from src.segmentation import UNet, evaluate_segmentation
    except Exception as e:
        print(f"Warning: failed to import segmentation components for U-Net recovery: {e}")
        return {}

    ckpt_candidates = []
    if "SEG_SAVE_PATH" in globals():
        try:
            ckpt_candidates.append(Path(SEG_SAVE_PATH))
        except Exception:
            pass

    model_dir = metrics_dir.parent.parent / "models" / "segmentation"
    # Prefer the standard checkpoint name used in this project.
    ckpt_candidates.extend([
        model_dir / "best_unet_real_mask.pth",
        model_dir / "best_unet.pth",
    ])

    ckpt_path = None
    seen = set()
    for cand in ckpt_candidates:
        cand = Path(cand)
        key = str(cand)
        if key in seen:
            continue
        seen.add(key)
        if cand.exists():
            ckpt_path = cand
            break

    if ckpt_path is None:
        print("Warning: U-Net metrics are missing and no U-Net checkpoint was found.")
        return {}

    print(f"Recovering U-Net metrics from checkpoint: {ckpt_path}")
    try:
        model = UNet(in_channels=1, out_channels=1, base_filters=64).to(DEVICE)
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        params = int(sum(p.numel() for p in model.parameters()))
        avg, _ = evaluate_segmentation(model, seg_test_loader, DEVICE)

        recovered = {
            "model": "U-Net",
            "checkpoint": str(ckpt_path),
            "params": params,
            "test_dice": float(avg.get("dice", np.nan)),
            "test_iou": float(avg.get("iou", np.nan)),
            "test_hd95": float(avg.get("hd95", avg.get("hausdorff95", np.nan))),
            "test_loss": float("nan"),
        }

        unet_metrics_path = metrics_dir / "unet_metrics.json"
        unet_metrics_path.write_text(json.dumps(recovered, indent=2))
        print("Saved recovered U-Net metrics to:", unet_metrics_path)
        return recovered
    except Exception as e:
        print(f"Warning: failed to recover U-Net metrics from {ckpt_path}: {e}")
        return {}


unet_metrics = _recover_unet_metrics_if_missing(_load_metrics(metrics_dir / "unet_metrics.json", "U-Net"))
resunet_metrics = _load_metrics(metrics_dir / "resunet_comparison.json", "ResUNet")
attn_metrics = _load_metrics(metrics_dir / "attention_unet_metrics.json", "Attention U-Net")

rows = [
    {
        "model": unet_metrics.get("model", "U-Net"),
        "test_dice": _metric(unet_metrics, ["test_dice", "dice"]),
        "test_iou": _metric(unet_metrics, ["test_iou", "iou"]),
        "test_hd95": _metric(unet_metrics, ["test_hd95", "hd95", "hausdorff95", "test_hausdorff95"]),
    },
    {
        "model": resunet_metrics.get("model", "ResUNet"),
        "test_dice": _metric(resunet_metrics, ["test_dice", "dice"]),
        "test_iou": _metric(resunet_metrics, ["test_iou", "iou"]),
        "test_hd95": _metric(resunet_metrics, ["test_hd95", "hd95", "hausdorff95", "test_hausdorff95"]),
    },
    {
        "model": attn_metrics.get("model", "Attention U-Net"),
        "test_dice": _metric(attn_metrics, ["test_dice", "dice"]),
        "test_iou": _metric(attn_metrics, ["test_iou", "iou"]),
        "test_hd95": _metric(attn_metrics, ["test_hd95", "hd95", "hausdorff95", "test_hausdorff95"]),
    },
]

df = pd.DataFrame(rows)
missing_mask = df[["test_dice", "test_iou", "test_hd95"]].isna().any(axis=1)
if missing_mask.any():
    print("Warning: missing metrics for:", ", ".join(df.loc[missing_mask, "model"].tolist()))

comparison_csv.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(comparison_csv, index=False)
display(df)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric, title in zip(
    axes,
    ["test_dice", "test_iou", "test_hd95"],
    ["Dice (higher better)", "IoU (higher better)", "HD95 (lower better)"],
):
    vals = pd.to_numeric(df[metric], errors="coerce").to_numpy(dtype=np.float64)
    finite = np.isfinite(vals)
    inf_mask = np.isinf(vals)
    nan_mask = np.isnan(vals)

    if finite.any():
        cap = float(np.max(vals[finite]))
        if cap <= 0:
            cap = 1.0
        cap *= 1.1
    else:
        cap = 1.0

    draw_vals = vals.copy()
    draw_vals[inf_mask] = cap
    draw_vals[nan_mask] = 0.0

    bars = ax.bar(df["model"], draw_vals)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)

    for b, v in zip(bars, vals):
        if np.isnan(v):
            lbl = "missing"
            b.set_alpha(0.35)
            b.set_hatch("//")
        elif np.isinf(v):
            lbl = "inf"
        else:
            lbl = f"{v:.2f}" if metric == "test_hd95" else f"{v:.4f}"
        ax.text(b.get_x() + b.get_width() / 2, b.get_height(), lbl, ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Single-figure comparison across 3 metrics (Dice/IoU/HD95)
# ============================================================
if "df" not in globals():
    raise RuntimeError("Run the metrics table cell above first so `df` is available.")

plot_df = df.copy()

metric_specs = [
    ("Dice", "test_dice", True),
    ("IoU", "test_iou", True),
    ("HD95", "test_hd95", False),  # lower is better -> invert after scaling
]

norm_data = {}
for label, col, higher_is_better in metric_specs:
    vals = pd.to_numeric(plot_df[col], errors="coerce").to_numpy(dtype=np.float64)
    finite = np.isfinite(vals)

    norm = np.full(vals.shape, np.nan, dtype=np.float64)
    if finite.any():
        vmin = vals[finite].min()
        vmax = vals[finite].max()

        if np.isclose(vmax, vmin):
            norm[finite] = 1.0
        else:
            scaled = (vals[finite] - vmin) / (vmax - vmin)
            norm[finite] = scaled if higher_is_better else (1.0 - scaled)

    norm_data[label] = norm

norm_df = pd.DataFrame(norm_data, index=plot_df["model"])

fig, ax = plt.subplots(figsize=(10, 5.5))
x = np.arange(len(norm_df.index))
width = 0.24

for i, metric in enumerate(norm_df.columns):
    vals = norm_df[metric].to_numpy(dtype=np.float64)
    draw_vals = np.nan_to_num(vals, nan=0.0)
    bars = ax.bar(x + (i - 1) * width, draw_vals, width=width, label=metric)

    for b, v in zip(bars, vals):
        txt = "missing" if np.isnan(v) else f"{v:.2f}"
        ax.text(
            b.get_x() + b.get_width() / 2,
            b.get_height() + 0.02,
            txt,
            ha="center",
            va="bottom",
            fontsize=9,
        )

ax.set_xticks(x)
ax.set_xticklabels(norm_df.index)
ax.set_ylim(0, 1.1)
ax.set_ylabel("Normalized score (0-1)")
ax.set_title("Model comparison in one chart (HD95 inverted: lower is better)")
ax.grid(axis="y", alpha=0.25)
ax.legend()

plt.tight_layout()
plt.show()

## 1.6 Post-Processing: Crop Tumor Regions for Classification

After segmentation, we:
1. Run the U-Net to predict a binary mask for each image
2. Find the largest connected component (tumor region)
3. Crop and resize to 224×224 for ResNet50 input

In [ ]:
# ============================================================
# Cell 1.6 - Generate classification patches using segmentation masks
# ============================================================
ROI_MARGIN = 20  # slightly larger context than before (was 10)


def _infer_patch_folder_name(model) -> str:
    """Choose output folder name from segmentation model type."""
    model_name = type(model).__name__.lower()
    if "resunet" in model_name:
        return "patches_resunet"
    if "unet" in model_name:
        return "patches_unet"
    return f"patches_{model_name}"


def generate_classification_patches(
    samples, seg_model, device, output_dir=None, output_size=(224, 224), margin=ROI_MARGIN,
):
    """Create cropped tumor patches for classification using segmentation predictions."""
    output_dir = Path(output_dir) if output_dir else PROCESSED_DIR / "patches"
    output_dir.mkdir(parents=True, exist_ok=True)

    patched_samples = []
    for img_path, label in tqdm(samples, desc="Creating patches"):
        img = np.array(Image.open(img_path).convert("L"))
        cls_name = CLASS_NAMES[label]

        if cls_name != "notumor" and seg_model is not None:
            mask = predict_mask(seg_model, img, device)
            if mask.sum() > 0:
                img_crop, _ = crop_tumor_region(img, mask, margin=margin, output_size=output_size)
            else:
                img_crop = np.array(Image.fromarray(img).resize(output_size))
        else:
            img_crop = np.array(Image.fromarray(img).resize(output_size))

        save_dir = output_dir / cls_name
        save_dir.mkdir(parents=True, exist_ok=True)
        save_path = save_dir / Path(img_path).name
        Image.fromarray(img_crop).save(str(save_path))
        patched_samples.append((str(save_path), label))

    return patched_samples


# Resolve segmentation model from session state.
candidate_model = globals().get("seg_model", globals().get("resunet_model", None))
if candidate_model is None or not hasattr(candidate_model, "eval"):
    raise RuntimeError("ResUNet model is not loaded. Run the segmentation load/train cell first.")
if "resunet" not in type(candidate_model).__name__.lower():
    raise RuntimeError(f"Expected ResUNet model, got: {type(candidate_model).__name__}")

seg_model = candidate_model
resunet_model = seg_model
patch_root = PROCESSED_DIR / _infer_patch_folder_name(seg_model)

# Skip if patches were already generated in a previous run for this model.
_patch_train_dir = patch_root / "train"
if _patch_train_dir.exists() and any(_patch_train_dir.rglob("*.jpg")):
    print(f"Patches already exist at {patch_root} - loading paths...")
    train_patches, val_patches, test_patches = [], [], []
    for split_name, split_data, out_list in [
        ("train", train_samples, train_patches),
        ("val", val_samples, val_patches),
        ("test", test_samples, test_patches),
    ]:
        _dir = patch_root / split_name
        for img_path, label in split_data:
            _p = _dir / CLASS_NAMES[label] / Path(img_path).name
            if _p.exists():
                out_list.append((str(_p), label))
            else:
                out_list.append((img_path, label))
    print(
        f"Loaded patches ({patch_root.name}): "
        f"train={len(train_patches)}, val={len(val_patches)}, test={len(test_patches)}"
    )
else:
    seg_model.eval()
    print(
        f"Generating cropped tumor patches using {type(seg_model).__name__} predictions "
        f"(margin={ROI_MARGIN}) into {patch_root}..."
    )
    train_patches = generate_classification_patches(
        train_samples, seg_model, DEVICE, patch_root / "train", margin=ROI_MARGIN
    )
    val_patches = generate_classification_patches(
        val_samples, seg_model, DEVICE, patch_root / "val", margin=ROI_MARGIN
    )
    test_patches = generate_classification_patches(
        test_samples, seg_model, DEVICE, patch_root / "test", margin=ROI_MARGIN
    )
    print(
        f"Patches created ({patch_root.name}): "
        f"train={len(train_patches)}, val={len(val_patches)}, test={len(test_patches)}"
    )

---
# Stage 2 – Classification (ResNet50)

Fine-tune an ImageNet-pretrained ResNet50 on the segmented tumor patches.

**Target metrics:** >95% accuracy, precision, recall, F1-score.

Features:
- Data augmentation (rotation, flip, brightness/contrast jitter)
- Class-weight balancing
- Grid-search hyperparameter optimisation
- Early stopping

## 2.1 Prepare DataLoaders

In [ ]:
# ============================================================
# Cell 2.1 - Build dual-input datasets & dataloaders
# ============================================================
import importlib, src.classification
importlib.reload(src.classification)
from src.classification import (
    build_resnet50, compute_class_weights,
    get_train_transforms, get_val_transforms,
    train_one_epoch, evaluate, train_classifier,
    grid_search_hyperparams,
)

from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
from pathlib import Path

INPUT_SIZE = cfg["classification"]["input_size"]
BATCH_SIZE = cfg["classification"]["batch_size"]
NUM_WORKERS = cfg["classification"]["num_workers"]
AUG_CFG = cfg["classification"]["augmentation"]


def _build_original_lookup(samples):
    """Map (label, filename) -> original image path for each split."""
    lookup = {}
    for img_path, label in samples:
        lookup[(int(label), Path(img_path).name)] = str(img_path)
    return lookup


class DualInputBrainTumorDataset(Dataset):
    """Return 6-channel input: [patch RGB (3), original RGB (3)]."""

    def __init__(self, patch_samples, original_lookup, patch_transform=None, original_transform=None):
        self.samples = [(str(p), int(l)) for p, l in patch_samples if Path(p).exists()]
        self.original_lookup = original_lookup
        self.patch_transform = patch_transform
        self.original_transform = original_transform if original_transform is not None else patch_transform
        self.to_tensor = transforms.ToTensor()

        dropped = len(patch_samples) - len(self.samples)
        if dropped > 0:
            print(f"Warning: dropped {dropped} missing patch files.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        patch_path, label = self.samples[idx]
        patch_img = Image.open(patch_path).convert("RGB")

        key = (label, Path(patch_path).name)
        original_path = self.original_lookup.get(key, patch_path)
        if not Path(original_path).exists():
            original_path = patch_path
        original_img = Image.open(original_path).convert("RGB")

        patch_tensor = self.patch_transform(patch_img) if self.patch_transform else self.to_tensor(patch_img)
        original_tensor = self.original_transform(original_img) if self.original_transform else self.to_tensor(original_img)
        dual_tensor = torch.cat([patch_tensor, original_tensor], dim=0)

        return dual_tensor, label


def _build_dual_input_tensor_from_patch(
    patch_path, label, original_lookup, patch_transform, original_transform,
):
    """Build a model-ready dual input tensor from one patch path."""
    patch_img = Image.open(patch_path).convert("RGB")
    key = (int(label), Path(patch_path).name)
    original_path = original_lookup.get(key, patch_path)
    if not Path(original_path).exists():
        original_path = patch_path
    original_img = Image.open(original_path).convert("RGB")

    patch_tensor = patch_transform(patch_img) if patch_transform else transforms.ToTensor()(patch_img)
    original_tensor = original_transform(original_img) if original_transform else transforms.ToTensor()(original_img)

    return torch.cat([patch_tensor, original_tensor], dim=0).unsqueeze(0), patch_img, original_img


train_original_lookup = _build_original_lookup(train_samples)
val_original_lookup = _build_original_lookup(val_samples)
test_original_lookup = _build_original_lookup(test_samples)

train_patch_tf = get_train_transforms(INPUT_SIZE, AUG_CFG)
eval_patch_tf = get_val_transforms(INPUT_SIZE)
original_tf = get_val_transforms(INPUT_SIZE)

train_ds = DualInputBrainTumorDataset(
    train_patches,
    train_original_lookup,
    patch_transform=train_patch_tf,
    original_transform=original_tf,
)
val_ds = DualInputBrainTumorDataset(
    val_patches,
    val_original_lookup,
    patch_transform=eval_patch_tf,
    original_transform=original_tf,
)
test_ds = DualInputBrainTumorDataset(
    test_patches,
    test_original_lookup,
    patch_transform=eval_patch_tf,
    original_transform=original_tf,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}  |  Test batches: {len(test_loader)}")

# Sanity check: expected shape = (batch, 6, H, W)
imgs, lbls = next(iter(train_loader))
print(f"Batch shape: {imgs.shape}, Labels: {lbls[:8].tolist()}")

## 2.2 Class-Weight Balancing

In [ ]:
# ============================================================
# Cell 2.2 – Compute class weights for imbalanced data
# ============================================================
train_labels = [s[1] for s in train_patches]
class_weights = compute_class_weights(train_labels, num_classes=cfg["dataset"]["num_classes"])
class_weights = class_weights.to(DEVICE)
print("Class weights:")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name:>12s}: {class_weights[i]:.4f}")

## 2.3 (Optional) Grid Search for Hyperparameters

In [ ]:
# ============================================================
# Cell 2.3 – Grid search (optional, takes time)
# ============================================================
import importlib, src.classification
importlib.reload(src.classification)
from src.classification import grid_search_hyperparams

RUN_GRID_SEARCH = False  # ← Set True only when you want to search again

if RUN_GRID_SEARCH:
    _train = [(p, l) for p, l in train_patches if Path(p).exists()]
    _val   = [(p, l) for p, l in val_patches   if Path(p).exists()]
    if len(_train) < len(train_patches) or len(_val) < len(val_patches):
        print(f"Filtered missing files: train {len(train_patches)}→{len(_train)}, val {len(val_patches)}→{len(_val)}")

    gs_results = grid_search_hyperparams(
        train_samples=_train,
        val_samples=_val,
        lr_candidates=cfg["classification"]["grid_search"]["learning_rates"],
        bs_candidates=cfg["classification"]["grid_search"]["batch_sizes"],
        device=DEVICE,
        epochs=15,
        num_classes=cfg["dataset"]["num_classes"],
        input_size=INPUT_SIZE,
        aug_cfg=AUG_CFG,
    )
    print(f"\nBest hyperparameters: {gs_results['best_params']}")

    # === AUTO UPDATE CONFIG WITH BEST PARAMS ===
    best = gs_results['best_params']
    cfg["classification"]["learning_rate"] = best["lr"]
    cfg["classification"]["batch_size"]    = best["bs"]
    print(f"Config automatically updated with best params → LR={best['lr']}, BS={best['bs']}")

else:
    cfg["classification"]["learning_rate"] = 5e-05   # ← this was the winner
    cfg["classification"]["batch_size"]    = 32
    print(f"  LR: {cfg['classification']['learning_rate']}")
    print(f"  Batch size: {cfg['classification']['batch_size']}")

## 2.4 Build Model & Train

In [ ]:
# ============================================================
# Cell 2.4 - Build dual-input ResNet50 classifier
# ============================================================
from torchvision import models


def build_resnet50_dual_input(num_classes=4, pretrained=True):
    """ResNet50 that accepts 6 channels: [patch RGB + original RGB]."""
    weights = models.ResNet50_Weights.DEFAULT if pretrained else None
    model = models.resnet50(weights=weights)

    old_conv = model.conv1
    new_conv = nn.Conv2d(
        6,
        old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=False,
    )

    with torch.no_grad():
        # Copy pretrained RGB filters to both channel groups and rescale.
        new_conv.weight[:, :3] = old_conv.weight
        new_conv.weight[:, 3:] = old_conv.weight
        new_conv.weight *= 0.5

    model.conv1 = new_conv
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


model = build_resnet50_dual_input(
    num_classes=cfg["dataset"]["num_classes"],
    pretrained=cfg["classification"]["pretrained"],
).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=cfg["classification"]["learning_rate"],
    weight_decay=cfg["classification"]["weight_decay"],
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=5
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:>12,}")
print(f"Trainable parameters: {trainable_params:>12,}")

In [ ]:
import importlib
import src.classification as classification
importlib.reload(classification)
from src.classification import train_classifier

In [ ]:
# ============================================================
# Cell 2.5 - Load trained dual-input ResNet50 (or train)
# ============================================================

if "DualInputBrainTumorDataset" not in globals():
    raise RuntimeError("Run Cell 2.1 first to define dual-input dataset utilities.")

# --- Rebuild loaders using dual-input dataset with safe path handling ---
_train_tf = get_train_transforms(INPUT_SIZE, AUG_CFG)
_val_tf = get_val_transforms(INPUT_SIZE)
_orig_tf = get_val_transforms(INPUT_SIZE)

_tds = DualInputBrainTumorDataset(
    train_patches,
    train_original_lookup,
    patch_transform=_train_tf,
    original_transform=_orig_tf,
)
_vds = DualInputBrainTumorDataset(
    val_patches,
    val_original_lookup,
    patch_transform=_val_tf,
    original_transform=_orig_tf,
)
print(f"Safe dual loaders: train={len(_tds)}, val={len(_vds)}")

train_loader = DataLoader(_tds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(_vds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
# --- end loaders ---

# ====================== Load saved model or train ======================
SAVE_PATH = CLS_MODEL_DIR / "best_resnet50_dual_input.pth"

LOAD_SAVED_MODEL = True  # Change to False only if you want to force retraining

if LOAD_SAVED_MODEL and SAVE_PATH.exists():
    print(f"Loading saved model from {SAVE_PATH}...")
    checkpoint = torch.load(SAVE_PATH, map_location=DEVICE, weights_only=True)

    # Handle both plain state_dict and full checkpoint formats
    try:
        if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
            model.load_state_dict(checkpoint["model_state_dict"])
            print("Loaded full checkpoint (including training metadata)")
        else:
            model.load_state_dict(checkpoint)
            print("Loaded weights directly")

        model = model.to(DEVICE)
        model.eval()  # Critical: switches to inference mode
        print("Model loaded successfully and set to evaluation mode.")
        history = None
    except RuntimeError as e:
        print(f"Checkpoint incompatible with current dual-input model. Retraining.\nDetails: {e}")
        history = train_classifier(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            device=DEVICE,
            epochs=cfg["classification"]["epochs"],
            patience=cfg["classification"]["patience"],
            save_path=str(SAVE_PATH),
        )
        print(f"Training complete. Best model saved to: {SAVE_PATH}")

else:
    print("No saved dual-input model found. Starting full training...")
    history = train_classifier(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        device=DEVICE,
        epochs=cfg["classification"]["epochs"],
        patience=cfg["classification"]["patience"],
        save_path=str(SAVE_PATH),
    )
    print(f"Training complete. Best model saved to: {SAVE_PATH}")

print(f"Current model path: {SAVE_PATH}")

In [ ]:
# ============================================================
# Cell 2.6 – Plot training curves
# ============================================================

import matplotlib.pyplot as plt
from PIL import Image

FIGURE_PATH = OUTPUT_DIR / "figures" / "training_curves.png"

if history is not None:
    print("Plotting training curves from current training run...")
    plot_training_curves(
        history,
        save_path=str(FIGURE_PATH),
    )
else:
    if FIGURE_PATH.exists():
        print(f"Loading saved training curves image from {FIGURE_PATH}")
        img = Image.open(FIGURE_PATH)
        plt.figure(figsize=(12, 8))
        plt.imshow(img)
        plt.axis('off')
        plt.title("Training and Validation Curves (Loaded Model)")
        plt.show()
    else:
        print("Warning: No history available and no saved training_curves.png found.")

## 2.7 Evaluate on Test Set

In [ ]:
# ============================================================
# Cell 2.7 – Load best model & evaluate on test set
# ============================================================
model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
model.eval()

test_loss, test_metrics, test_cm = evaluate(model, test_loader, criterion, DEVICE)

print("\n" + "=" * 50)
print("       TEST SET RESULTS")
print("=" * 50)
print(f"  Loss:      {test_loss:.4f}")
print(f"  Accuracy:  {test_metrics['accuracy']:.4f}")
print(f"  Precision: {test_metrics['precision']:.4f}")
print(f"  Recall:    {test_metrics['recall']:.4f}")
print(f"  F1-Score:  {test_metrics['f1']:.4f}")
print("=" * 50)

In [ ]:
# ============================================================
# Cell 2.8 – Confusion matrix
# ============================================================
plot_confusion_matrix(
    test_cm,
    class_names=CLASS_NAMES,
    title="Test Set Confusion Matrix",
    save_path=str(OUTPUT_DIR / "figures" / "confusion_matrix.png"),
)

# Per-class metrics
from sklearn.metrics import classification_report
all_preds, all_labels = [], []
model.eval()
with torch.no_grad():
    for imgs, lbls in test_loader:
        preds = model(imgs.to(DEVICE)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(lbls.numpy())

print("\nPer-class classification report:")
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4))

In [ ]:
# ============================================================
# Cell 2.9 - Test metric comparison for 3 classification methods
# ============================================================
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt


project_root = Path(PROJECT_ROOT) if "PROJECT_ROOT" in globals() else Path.cwd()
fig_dir = (Path(OUTPUT_DIR) / "figures") if "OUTPUT_DIR" in globals() else (project_root / "outputs" / "figures")


def _safe_float(v):
    try:
        return float(v)
    except Exception:
        return np.nan


def _load_json(path: Path):
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text())
    except Exception:
        return None


def _pack_metrics(loss, accuracy, f1):
    return {
        "test_loss": _safe_float(loss),
        "accuracy": _safe_float(accuracy),
        "f1": _safe_float(f1),
    }


def _extract_from_payload(payload):
    if not isinstance(payload, dict):
        return _pack_metrics(np.nan, np.nan, np.nan)
    return _pack_metrics(
        payload.get("test_loss", payload.get("loss", np.nan)),
        payload.get("accuracy", np.nan),
        payload.get("f1", payload.get("f1_score", np.nan)),
    )


# Resolve metrics for each method, with explicit source tracking.
metrics_by_method = {}
source_by_method = {}

# 1) ResNet50 (Original Input)
orig = None
for gv in ["test_metrics_original", "test_metrics_pure", "pure_test_metrics"]:
    if gv in globals() and isinstance(globals()[gv], dict):
        _m = globals()[gv]
        _loss = globals().get("test_loss_original", globals().get("test_loss_pure", np.nan))
        orig = _pack_metrics(_loss, _m.get("accuracy", np.nan), _m.get("f1", np.nan))
        source_by_method["ResNet50 (Original Input)"] = f"global:{gv}"
        break
if orig is None:
    orig_file = project_root / "models" / "classification" / "pure_baseline" / "test_metrics.json"
    payload = _load_json(orig_file)
    if payload is not None:
        orig = _extract_from_payload(payload)
        source_by_method["ResNet50 (Original Input)"] = str(orig_file.relative_to(project_root))
if orig is None:
    orig = _pack_metrics(np.nan, np.nan, np.nan)
    source_by_method["ResNet50 (Original Input)"] = "missing"
metrics_by_method["ResNet50 (Original Input)"] = orig

# 2) ResNet50 (Cropped Input)
crop = None
for gv in ["test_metrics_single", "single_test_metrics", "test_metrics_cropped"]:
    if gv in globals() and isinstance(globals()[gv], dict):
        _m = globals()[gv]
        _loss = globals().get("test_loss_single", globals().get("test_loss_cropped", np.nan))
        crop = _pack_metrics(_loss, _m.get("accuracy", np.nan), _m.get("f1", np.nan))
        source_by_method["ResNet50 (Cropped Input)"] = f"global:{gv}"
        break
if crop is None:
    crop_candidates = [
        project_root / "outputs" / "metrics" / "resnet50_cropped_test_metrics.json",
        project_root / "outputs" / "metrics" / "test_metrics_resnet50_cropped.json",
        project_root / "models" / "classification" / "cropped" / "test_metrics.json",
        project_root / "models" / "classification" / "resnet50_single" / "test_metrics.json",
    ]
    for p in crop_candidates:
        payload = _load_json(p)
        if payload is not None:
            crop = _extract_from_payload(payload)
            source_by_method["ResNet50 (Cropped Input)"] = str(p.relative_to(project_root))
            break
if crop is None:
    crop = _pack_metrics(np.nan, np.nan, np.nan)
    source_by_method["ResNet50 (Cropped Input)"] = "missing"
metrics_by_method["ResNet50 (Cropped Input)"] = crop

# 3) ResNet50 (Dual Input)
dual = None
if "test_metrics" in globals() and isinstance(test_metrics, dict):
    dual = _pack_metrics(globals().get("test_loss", np.nan), test_metrics.get("accuracy", np.nan), test_metrics.get("f1", np.nan))
    source_by_method["ResNet50 (Dual Input)"] = "global:test_metrics"
if dual is None:
    dual_candidates = [
        project_root / "outputs" / "metrics" / "resnet50_dual_test_metrics.json",
        project_root / "outputs" / "metrics" / "test_metrics_resnet50_dual_input.json",
        project_root / "models" / "classification" / "dual" / "test_metrics.json",
    ]
    for p in dual_candidates:
        payload = _load_json(p)
        if payload is not None:
            dual = _extract_from_payload(payload)
            source_by_method["ResNet50 (Dual Input)"] = str(p.relative_to(project_root))
            break
if dual is None:
    # Fallback: results_summary.csv may have accuracy/F1 but usually no test loss.
    rs = project_root / "outputs" / "results_summary.csv"
    if rs.exists():
        try:
            import pandas as pd
            _df = pd.read_csv(rs)
            acc = _df.loc[(_df["Stage"] == "Classification") & (_df["Metric"] == "Accuracy"), "Value"]
            f1 = _df.loc[(_df["Stage"] == "Classification") & (_df["Metric"] == "F1-Score"), "Value"]
            dual = _pack_metrics(np.nan, acc.iloc[0] if len(acc) else np.nan, f1.iloc[0] if len(f1) else np.nan)
            source_by_method["ResNet50 (Dual Input)"] = "outputs/results_summary.csv (loss missing)"
        except Exception:
            dual = None
if dual is None:
    dual = _pack_metrics(np.nan, np.nan, np.nan)
    source_by_method["ResNet50 (Dual Input)"] = "missing"
metrics_by_method["ResNet50 (Dual Input)"] = dual


# Build table + missing-information report
metric_order = ["f1", "accuracy", "test_loss"]
metric_labels = ["F1", "Accuracy", "Test Loss"]
methods = list(metrics_by_method.keys())

print("Metric sources:")
for m in methods:
    print(f"  - {m}: {source_by_method.get(m, 'missing')}")

missing_report = {}
for m in methods:
    miss = [k for k in metric_order if not np.isfinite(metrics_by_method[m][k])]
    if miss:
        missing_report[m] = miss

if missing_report:
    print("\nMissing information detected:")
    for m, miss in missing_report.items():
        print(f"  - {m}: {', '.join(miss)}")
else:
    print("\nAll required test metrics are available.")

if all(all(not np.isfinite(metrics_by_method[m][k]) for k in metric_order) for m in methods):
    raise RuntimeError("No test metrics available for plotting.")


# One graph: grouped bars (F1, Accuracy, Test Loss) x 3 methods
x = np.arange(len(metric_order))
width = 0.24
offsets = [-width, 0.0, width]
colors = ["#4C78A8", "#F58518", "#54A24B"]

fig, ax = plt.subplots(figsize=(11, 6))
for idx, method in enumerate(methods):
    vals = [metrics_by_method[method][k] for k in metric_order]
    draw_vals = [v if np.isfinite(v) else 0.0 for v in vals]
    bars = ax.bar(x + offsets[idx], draw_vals, width=width, label=method, color=colors[idx], alpha=0.9)

    for b, v in zip(bars, vals):
        if np.isfinite(v):
            ax.text(b.get_x() + b.get_width()/2, b.get_height(), f"{v:.4f}", ha="center", va="bottom", fontsize=9)
        else:
            b.set_hatch("//")
            b.set_alpha(0.35)
            ax.text(b.get_x() + b.get_width()/2, 0.01, "missing", ha="center", va="bottom", fontsize=8, rotation=90)

ax.set_xticks(x)
ax.set_xticklabels(metric_labels)
ax.set_ylabel("Metric Value")
ax.set_title("Test Metrics Comparison: ResNet50 Original vs Cropped vs Dual")
ax.grid(axis="y", alpha=0.25)
ax.legend(fontsize=9)

plt.tight_layout()
fig_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_dir / "classification_test_metrics_three_methods.png", dpi=160, bbox_inches="tight")
plt.show()

---
# Stage 3 – Explainability & Report Generation

This stage generates:
1. **Grad-CAM heatmaps** – visual explanations of which regions influenced the classifier
2. **VLM reports** – natural-language medical reports describing the finding

Both outputs are overlaid on / associated with the original MRI for clinical interpretability.

## 3.1 Grad-CAM Heatmap Generation

In [ ]:
!pip install sacremoses

In [ ]:
import os
from huggingface_hub import login, whoami

hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
if hf_token:
    login(token=hf_token)
else:
    print("HF_TOKEN not set; skipping Hugging Face login")


In [ ]:
import importlib
import src.explainability as ex

print("module path:", ex.__file__)
importlib.reload(ex)

from src.explainability import CLIPReportGenerator
print("has reset method:", hasattr(CLIPReportGenerator, "reset_generation_state"))

# Recreate vlm from the reloaded class
vlm = CLIPReportGenerator(
    device="auto",
    decoder_model_name="google/medgemma-1.5-4b-it"
)

vlm.reset_generation_state(unload_model=True)
vlm.load_model()

In [ ]:
print(vlm._generation_unavailable)
# ============================================================
# Cell 3.1 - Generate Grad-CAM heatmaps on test samples
# ============================================================
from src.explainability import generate_gradcam, overlay_gradcam, CLIPReportGenerator, compute_stage_consistency_iou

model.eval()
target_layer = model.layer4[-1]  # last bottleneck block in layer4

seg_report_model = globals().get("seg_model", globals().get("resunet_model", None))
if seg_report_model is None or not hasattr(seg_report_model, "eval"):
    raise RuntimeError("ResUNet model is not available. Run the segmentation train/load cell first.")
seg_report_model.eval()

patch_transform = get_val_transforms(INPUT_SIZE)
original_transform = get_val_transforms(INPUT_SIZE)

if "test_original_lookup" not in globals():
    test_original_lookup = _build_original_lookup(test_samples)

num_examples = min(8, len(test_patches))
fig, axes = plt.subplots(num_examples, 4, figsize=(20, 5 * num_examples))
if num_examples == 1:
    axes = axes[np.newaxis, :]

for i in range(num_examples):
    img_path, true_label = test_patches[i]

    # Build dual input = [patch RGB, original RGB]
    input_tensor, patch_img, original_img = _build_dual_input_tensor_from_patch(
        img_path,
        true_label,
        test_original_lookup,
        patch_transform,
        original_transform,
    )
    patch_arr = np.array(patch_img.resize((INPUT_SIZE, INPUT_SIZE)))

    # Predict
    with torch.no_grad():
        logits = model(input_tensor.to(DEVICE))
        probs = torch.softmax(logits, dim=1)
        pred_class = probs.argmax(1).item()
        confidence = probs[0, pred_class].item()

    # Grad-CAM
    heatmap = generate_gradcam(model, input_tensor, target_layer, DEVICE, pred_class)
    overlay = overlay_gradcam(patch_arr, heatmap)

    # Segmentation mask for stage-consistency IoU
    gray_patch = np.array(patch_img.convert("L"))
    stage_mask_raw = predict_mask(seg_report_model, gray_patch, DEVICE)
    stage_mask = np.array(Image.fromarray(stage_mask_raw).resize((INPUT_SIZE, INPUT_SIZE), Image.NEAREST))
    stage_iou = compute_stage_consistency_iou(stage_mask, heatmap)

    # Plot
    axes[i, 0].imshow(patch_arr)
    axes[i, 0].set_title(f"Patch (GT: {CLASS_NAMES[true_label]})")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(heatmap, cmap="jet")
    axes[i, 1].set_title("Grad-CAM Heatmap")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(overlay)
    axes[i, 2].set_title(f"Overlay (Pred: {CLASS_NAMES[pred_class]}, {confidence:.1%})")
    axes[i, 2].axis("off")

    axes[i, 3].axis("off")
    report = vlm.generate_report(
        original_img,
        pred_class,
        confidence,
        heatmap,
        segmentation_mask=stage_mask,
        stage_iou=stage_iou,
    )
    axes[i, 3].text(
        0.05,
        0.95,
        report,
        transform=axes[i, 3].transAxes,
        fontsize=8,
        verticalalignment="top",
        wrap=True,
        family="serif",
    )
    axes[i, 3].set_title("VLM Report")

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "gradcam" / "gradcam_examples.png"), dpi=150, bbox_inches="tight")
plt.show()

## 3.2 CLIP-based VLM Report Generation

In [ ]:
# ============================================================
# Cell 3.2 - VLM report generation (template + CLIP embedding)
# ============================================================
from src.explainability import compute_stage_consistency_iou

vlm = CLIPReportGenerator(
    device=str( ),
    # model_name=cfg["explainability"]["vlm_model"],
)

if "test_original_lookup" not in globals():
    test_original_lookup = _build_original_lookup(test_samples)

patch_transform = get_val_transforms(INPUT_SIZE)
original_transform = get_val_transforms(INPUT_SIZE)

seg_report_model = globals().get("seg_model", globals().get("resunet_model", None))
if seg_report_model is None or not hasattr(seg_report_model, "eval"):
    raise RuntimeError("ResUNet model is not available. Run the segmentation train/load cell first.")
seg_report_model.eval()

# Generate reports for a few test examples
print("=" * 70)
print("SAMPLE VLM REPORTS")
print("=" * 70)

for i in range(min(4, len(test_patches))):
    img_path, true_label = test_patches[i]
    input_tensor, patch_img, original_img = _build_dual_input_tensor_from_patch(
        img_path,
        true_label,
        test_original_lookup,
        patch_transform,
        original_transform,
    )

    with torch.no_grad():
        logits = model(input_tensor.to(DEVICE))
        probs = torch.softmax(logits, dim=1)
        pred_class = probs.argmax(1).item()
        confidence = probs[0, pred_class].item()

    heatmap = generate_gradcam(model, input_tensor, target_layer, DEVICE, pred_class)

    gray_patch = np.array(patch_img.convert("L"))
    stage_mask_raw = predict_mask(seg_report_model, gray_patch, DEVICE)
    stage_mask = np.array(Image.fromarray(stage_mask_raw).resize((INPUT_SIZE, INPUT_SIZE), Image.NEAREST))
    stage_iou = compute_stage_consistency_iou(stage_mask, heatmap)

    report = vlm.generate_report(
        image=original_img,
        predicted_class=pred_class,
        confidence=confidence,
        heatmap=heatmap,
        segmentation_mask=stage_mask,
        stage_iou=stage_iou,
    )

    print(f"\n--- Sample {i+1} ---")
    print(f"True: {CLASS_NAMES[true_label]} | Predicted: {CLASS_NAMES[pred_class]} | Conf: {confidence:.1%}")
    print(report)
    print()

---
# Stage 4 – Qualitative Results Panel

This section produces the comprehensive 5-panel visualisation matching the paper's
"Preliminary Results" section:

**Original MRI | Ground-Truth Mask | Predicted Mask | Grad-CAM | VLM Report**

In [ ]:
# ============================================================
# Cell 4.1 - Full qualitative results panel
# ============================================================
from collections import defaultdict
from importlib import reload
from pathlib import Path
import inspect
import pickle
import random
import re

try:
    import cv2
except ImportError:
    cv2 = None

import src.explainability as explainability_mod
import src.visualization as visualization_mod
reload(explainability_mod)
reload(visualization_mod)
from src.explainability import CLIPReportGenerator, compute_stage_consistency_iou, summarize_report_text
from src.visualization import plot_qualitative_panel as plot_qualitative_panel_reloaded

# Ensure we use the reloaded plotting function in this cell scope.
plot_qualitative_panel = plot_qualitative_panel_reloaded

# Rebuild VLM object when class definition changed in-memory.
if "vlm" in globals() and not isinstance(vlm, CLIPReportGenerator):
    vlm = CLIPReportGenerator(
        device=(getattr(vlm, "device", "auto") or "auto"),
        decoder_model_name=getattr(vlm, "decoder_model_name", "google/medgemma-1.5-4b-it"),
        max_new_tokens=getattr(vlm, "max_new_tokens", 220),
        temperature=getattr(vlm, "temperature", 0.2),
        repetition_penalty=getattr(vlm, "repetition_penalty", 1.3),
    )

model.eval()

seg_panel_model = globals().get("seg_model", globals().get("resunet_model", None))
if seg_panel_model is None or not hasattr(seg_panel_model, "eval"):
    raise RuntimeError("ResUNet model is not available. Run the segmentation train/load cell first.")
if "resunet" not in type(seg_panel_model).__name__.lower():
    raise RuntimeError(f"Expected ResUNet model, got: {type(seg_panel_model).__name__}")

seg_panel_model.eval()

if "test_original_lookup" not in globals():
    test_original_lookup = _build_original_lookup(test_samples)


def _normalize_key(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(text).lower())


def _class_hint_from_path(path_like: str) -> str:
    p = str(path_like).lower()
    if "meningioma" in p:
        return "meningioma"
    if "pituitary" in p:
        return "pituitary"
    if "glioma" in p:
        return "glioma"
    if "no_tumor" in p or "notumor" in p:
        return "notumor"
    return ""


def _build_seg_gt_lookup(seg_pairs):
    """Build robust filename/stem lookup from segmentation image-mask pairs."""
    by_name, by_stem, by_norm, by_digits = {}, {}, {}, {}
    for img_p, mask_p in seg_pairs:
        if not Path(mask_p).exists():
            continue
        img_p = str(img_p)
        mask_p = str(mask_p)
        name = Path(img_p).name.lower()
        stem = Path(img_p).stem.lower()
        norm = _normalize_key(stem)
        digit_match = re.search(r"(\d{4,})", stem)
        class_hint = _class_hint_from_path(img_p)
        entry = {
            "image_path": img_p,
            "mask_path": mask_p,
            "class_hint": class_hint,
        }

        by_name.setdefault(name, []).append(entry)
        by_stem.setdefault(stem, []).append(entry)
        if norm:
            by_norm.setdefault(norm, []).append(entry)
        if digit_match:
            by_digits.setdefault(digit_match.group(1), []).append(entry)
    return by_name, by_stem, by_norm, by_digits


def _collect_all_seg_pairs():
    """Union of segmentation splits + saved pickles. Classification filenames often only match here."""
    merged: list = []
    seen: set = set()

    def _normalize_pair(a, b):
        a = str(a)
        b = str(b)
        a_is_mask = "_mask" in Path(a).stem.lower() or "mask" in Path(a).name.lower()
        b_is_mask = "_mask" in Path(b).stem.lower() or "mask" in Path(b).name.lower()
        if a_is_mask and not b_is_mask:
            return b, a
        return a, b

    def _add(lst):
        if not lst:
            return
        for t in lst:
            if not (isinstance(t, (tuple, list)) and len(t) == 2):
                continue
            img_p, mask_p = _normalize_pair(t[0], t[1])
            sig = (str(Path(img_p).resolve()), str(Path(mask_p).resolve()))
            if sig not in seen:
                seen.add(sig)
                merged.append((img_p, mask_p))

    if "pairs" in globals():
        _add(globals()["pairs"])
    for key in ("seg_train", "seg_val", "seg_test"):
        if key in globals():
            _add(globals()[key])

    if not merged:
        proc = globals().get("PROCESSED_DIR")
        if proc is not None:
            proc_path = Path(proc)
            for pickle_name in ("seg_train_pairs.pkl", "seg_val_pairs.pkl", "seg_test_pairs.pkl"):
                pkl = proc_path / pickle_name
                if pkl.exists():
                    with open(pkl, "rb") as f:
                        _add(pickle.load(f))
    return merged


def _load_gray_image_safe(path_like: str):
    try:
        p = Path(str(path_like))
        if not p.exists():
            return None
        return np.array(Image.open(p).convert("L"))
    except Exception:
        return None


def _resize_gray_for_match(arr: np.ndarray, out_hw: tuple) -> np.ndarray:
    target_h, target_w = int(out_hw[0]), int(out_hw[1])
    gray = np.asarray(arr, dtype=np.float32).squeeze()
    if gray.shape == (target_h, target_w):
        return gray
    if cv2 is not None:
        return cv2.resize(gray, (target_w, target_h), interpolation=cv2.INTER_LINEAR).astype(np.float32)
    resized = np.array(
        Image.fromarray(np.clip(gray, 0, 255).astype(np.uint8)).resize((target_w, target_h), Image.BILINEAR),
        dtype=np.float32,
    )
    return resized


def _image_similarity_score(ref_gray: np.ndarray, cand_gray: np.ndarray) -> float:
    """Return normalized cross-correlation score in [-1, 1]."""
    ref = np.asarray(ref_gray, dtype=np.float32).squeeze()
    cand = np.asarray(cand_gray, dtype=np.float32).squeeze()
    if ref.ndim != 2 or cand.ndim != 2 or ref.size == 0 or cand.size == 0:
        return -1.0

    cand = _resize_gray_for_match(cand, ref.shape)
    ref = np.nan_to_num(ref, nan=0.0, posinf=0.0, neginf=0.0)
    cand = np.nan_to_num(cand, nan=0.0, posinf=0.0, neginf=0.0)

    ref = (ref - ref.mean()) / (ref.std() + 1e-6)
    cand = (cand - cand.mean()) / (cand.std() + 1e-6)

    ncc = float(np.mean(ref * cand))
    ncc_flip = float(np.mean(ref * np.fliplr(cand)))
    return max(ncc, ncc_flip)


def _resolve_seg_mask_path(
    original_img_path: str,
    by_name: dict,
    by_stem: dict,
    by_norm: dict,
    by_digits: dict,
    expected_class: str = "",
    original_gray: np.ndarray = None,
) -> str:
    """Resolve GT mask with class-aware matching and image-similarity verification."""
    p = Path(original_img_path)
    name_key = p.name.lower()
    stem_key = p.stem.lower()
    norm_key = _normalize_key(stem_key)
    digit_match = re.search(r"(\d{4,})", stem_key)

    candidate_pool = []

    def _extend(entries, rank: int):
        if not entries:
            return
        for e in entries:
            if not isinstance(e, dict) or "mask_path" not in e:
                continue
            candidate_pool.append((e, int(rank)))

    _extend(by_name.get(name_key), rank=0)
    _extend(by_stem.get(stem_key), rank=1)
    for ext in (".png", ".jpg", ".jpeg", ".tif"):
        _extend(by_name.get(f"{stem_key}{ext}"), rank=1)
    _extend(by_norm.get(norm_key), rank=2)
    if digit_match:
        _extend(by_digits.get(digit_match.group(1)), rank=3)

    # De-duplicate candidates by mask path, keeping the best (lowest) rank.
    dedup = {}
    for e, rank in candidate_pool:
        mp = str(e.get("mask_path", ""))
        if not mp:
            continue
        if mp not in dedup or rank < dedup[mp]["rank"]:
            dedup[mp] = {"entry": e, "rank": rank}
    candidates = list(dedup.values())

    if expected_class and expected_class != "notumor":
        class_matched = [
            c for c in candidates
            if not c["entry"].get("class_hint") or c["entry"].get("class_hint") == expected_class
        ]
        if class_matched:
            candidates = class_matched

    if not candidates:
        return ""

    ref_gray = original_gray
    if ref_gray is None:
        ref_gray = _load_gray_image_safe(original_img_path)

    if ref_gray is None:
        strict = sorted((c for c in candidates if c["rank"] <= 1), key=lambda c: c["rank"])
        if strict:
            return str(strict[0]["entry"]["mask_path"])
        return ""

    scored = []
    for c in candidates:
        seg_img_path = c["entry"].get("image_path", "")
        cand_gray = _load_gray_image_safe(seg_img_path)
        if cand_gray is None:
            continue
        score = _image_similarity_score(ref_gray, cand_gray)
        scored.append({"entry": c["entry"], "rank": c["rank"], "score": score})

    if not scored:
        return ""

    scored.sort(key=lambda x: (-float(x["score"]), int(x["rank"])))
    best = scored[0]
    best_score = float(best["score"])
    best_rank = int(best["rank"])

    # Require stronger agreement for looser key matches.
    if best_rank <= 1:
        min_score = 0.15
    elif best_rank == 2:
        min_score = 0.35
    else:
        min_score = 0.55

    if best_score < min_score:
        return ""

    return str(best["entry"]["mask_path"])


_all_seg_pairs = _collect_all_seg_pairs()
seg_gt_by_name, seg_gt_by_stem, seg_gt_by_norm, seg_gt_by_digits = _build_seg_gt_lookup(_all_seg_pairs)


def _norm_path_str(path_like: str) -> str:
    try:
        return str(Path(str(path_like)).resolve())
    except Exception:
        return str(path_like)


seg_mask_to_img = {}
for _img_p, _mask_p in _all_seg_pairs:
    seg_mask_to_img[_norm_path_str(_mask_p)] = str(_img_p)


def _matched_seg_image_path(mask_path: str) -> str:
    if not mask_path:
        return ""
    return seg_mask_to_img.get(_norm_path_str(mask_path), "")


if seg_gt_by_name:
    print(f"GT mask lookup: {len(seg_gt_by_name)} unique image names from {len(_all_seg_pairs)} seg pairs")
else:
    print(
        "GT mask lookup empty — run Stage-1 segmentation cells (pairs / seg_* splits or seg_*_pairs.pkl). "
        "Tumor GT overlays will be shown as unavailable when no dataset mask match exists."
    )

patch_transform = get_val_transforms(INPUT_SIZE)
original_transform = get_val_transforms(INPUT_SIZE)


def _refine_panel_mask(mask: np.ndarray) -> np.ndarray:
    """Refine binary mask for qualitative display (close gaps + denoise)."""
    mask_bin = (np.asarray(mask) > 0).astype(np.uint8)
    if mask_bin.sum() == 0:
        return mask_bin
    if cv2 is None:
        return mask_bin

    mask_bin = cv2.morphologyEx(mask_bin, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8), iterations=1)
    mask_bin = cv2.morphologyEx(mask_bin, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8), iterations=1)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_bin, connectivity=8)
    if num_labels > 1:
        largest_label = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
        mask_bin = (labels == largest_label).astype(np.uint8)

    # Slightly expand borders to reduce obvious under-coverage in visualization.
    mask_bin = cv2.dilate(mask_bin, np.ones((3, 3), np.uint8), iterations=1)
    return mask_bin.astype(np.uint8)


def _resize_binary_mask_to_shape(mask: np.ndarray, out_hw: tuple) -> np.ndarray:
    """Resize binary mask into target (H, W) using nearest-neighbor semantics."""
    mask_bin = (np.asarray(mask) > 0).astype(np.uint8)
    target_h, target_w = int(out_hw[0]), int(out_hw[1])
    if mask_bin.shape == (target_h, target_w):
        return mask_bin
    resized = np.array(
        Image.fromarray((mask_bin * 255).astype(np.uint8)).resize((target_w, target_h), Image.NEAREST)
    )
    return (resized > 127).astype(np.uint8)


# Pick a broader and more diverse panel, but prioritize samples with dataset GT masks.
# Sampling controls:
# - force_new_panels=True: sample without replacement across reruns (until pool exhausted).
# - panel_random_seed=None: use true random seed each run; set an int for reproducible panels.
force_new_panels = True
panel_random_seed = None

max_panels = min(8, len(test_patches))
run_seed = int(panel_random_seed) if panel_random_seed is not None else random.SystemRandom().randint(0, 2**32 - 1)
rng = random.Random(run_seed)
print(
    f"Qualitative panel random seed: {run_seed} | "
    f"force_new_panels={force_new_panels}"
)


def _has_dataset_gt_for_panel_index(sample_idx: int) -> bool:
    patch_path, label = test_patches[sample_idx]
    cls_name = CLASS_NAMES[int(label)]
    if cls_name == "notumor":
        return True
    key = (int(label), Path(patch_path).name)
    original_path = str(test_original_lookup.get(key, patch_path))
    original_gray = _load_gray_image_safe(original_path)
    gt_path = _resolve_seg_mask_path(
        original_path,
        seg_gt_by_name,
        seg_gt_by_stem,
        seg_gt_by_norm,
        seg_gt_by_digits,
        expected_class=cls_name,
        original_gray=original_gray,
    )
    return bool(gt_path and Path(gt_path).exists())


label_to_indices_with_gt = defaultdict(list)
label_to_indices_without_gt = defaultdict(list)
seen_original_paths = set()

for idx, (patch_path, label) in enumerate(test_patches):
    key = (int(label), Path(patch_path).name)
    original_path = str(test_original_lookup.get(key, patch_path))
    if original_path in seen_original_paths:
        continue
    seen_original_paths.add(original_path)

    if _has_dataset_gt_for_panel_index(idx):
        label_to_indices_with_gt[int(label)].append(idx)
    else:
        label_to_indices_without_gt[int(label)].append(idx)

labels_present = sorted(set(label_to_indices_with_gt.keys()) | set(label_to_indices_without_gt.keys()))
candidate_by_label = {}
for lbl in labels_present:
    with_gt = list(dict.fromkeys(label_to_indices_with_gt.get(lbl, [])))
    without_gt = [i for i in label_to_indices_without_gt.get(lbl, []) if i not in with_gt]
    candidate_by_label[lbl] = {
        "with_gt": with_gt,
        "without_gt": without_gt,
    }

candidate_with_gt = [i for bucket in candidate_by_label.values() for i in bucket["with_gt"]]
candidate_without_gt = [i for bucket in candidate_by_label.values() for i in bucket["without_gt"]]
all_candidate_indices = set(candidate_with_gt) | set(candidate_without_gt)

pool_class_stats = ", ".join(
    f"{CLASS_NAMES[lbl]}={len(candidate_by_label[lbl]['with_gt']) + len(candidate_by_label[lbl]['without_gt'])}"
    for lbl in labels_present
)
print(
    f"Candidate pool: total={len(all_candidate_indices)}, "
    f"with_gt={len(candidate_with_gt)}, without_gt={len(candidate_without_gt)}"
)
if pool_class_stats:
    print(f"Candidate pool by class: {pool_class_stats}")

pool_signature = tuple(sorted(all_candidate_indices))
prev_pool_signature = globals().get("_panel_candidate_pool_signature")
if prev_pool_signature != pool_signature:
    globals()["_panel_used_indices_history"] = []

used_history = set(globals().get("_panel_used_indices_history", [])) if force_new_panels else set()


def _available_by_label(used_set: set):
    avail = {}
    total = 0
    for lbl, bucket in candidate_by_label.items():
        with_avail = [i for i in bucket["with_gt"] if i not in used_set]
        without_avail = [i for i in bucket["without_gt"] if i not in used_set]
        if with_avail or without_avail:
            avail[lbl] = {"with_gt": with_avail, "without_gt": without_avail}
            total += len(with_avail) + len(without_avail)
    return avail, total


def _draw_from_label_bucket(local_rng: random.Random, bucket: dict):
    pool = bucket["with_gt"] if bucket["with_gt"] else bucket["without_gt"]
    if not pool:
        return None
    pick_pos = local_rng.randrange(len(pool))
    return pool.pop(pick_pos)


def _sample_indices(local_rng: random.Random, used_set: set) -> list:
    available_by_label, total_available = _available_by_label(used_set)

    if force_new_panels and total_available < max_panels:
        # Exhausted current cycle -> start a new cycle.
        used_set.clear()
        available_by_label, total_available = _available_by_label(used_set)
        print("Panel history exhausted; reset history for a fresh cycle.")

    selected = []
    active_labels = [lbl for lbl, bucket in available_by_label.items() if bucket["with_gt"] or bucket["without_gt"]]
    local_rng.shuffle(active_labels)

    # Class-balanced round-robin: at most one sample per class each pass.
    while len(selected) < max_panels and active_labels:
        made_pick = False
        for lbl in list(active_labels):
            bucket = available_by_label[lbl]
            picked = _draw_from_label_bucket(local_rng, bucket)
            if picked is not None:
                selected.append(picked)
                made_pick = True
            if not bucket["with_gt"] and not bucket["without_gt"]:
                active_labels.remove(lbl)
            if len(selected) >= max_panels:
                break
        if not made_pick:
            break
        local_rng.shuffle(active_labels)

    if not selected:
        fallback = list(range(len(test_patches)))
        local_rng.shuffle(fallback)
        selected = fallback[: min(max_panels, len(fallback))]

    local_rng.shuffle(selected)
    return selected


selected_indices = _sample_indices(rng, used_history)

prev_selection_sig = globals().get("_last_panel_selection_signature")
curr_selection_sig = tuple(sorted(selected_indices))
has_alternatives = len(all_candidate_indices) > len(selected_indices)

if force_new_panels and prev_selection_sig is not None and curr_selection_sig == prev_selection_sig and has_alternatives:
    # Strong retry loop to avoid identical consecutive panel sets.
    for _ in range(128):
        retry_seed = random.SystemRandom().randint(0, 2**32 - 1)
        retry_rng = random.Random(retry_seed)
        alt_indices = _sample_indices(retry_rng, set(used_history))
        alt_sig = tuple(sorted(alt_indices))
        if alt_sig != prev_selection_sig:
            selected_indices = alt_indices
            run_seed = retry_seed
            curr_selection_sig = alt_sig
            print(f"Force-resampled panels with new seed: {run_seed}")
            break
    else:
        print("force_new_panels=True but no alternative panel set found; keeping current selection.")

if force_new_panels:
    used_history.update(selected_indices)
    globals()["_panel_used_indices_history"] = sorted(used_history)

globals()["_panel_candidate_pool_signature"] = pool_signature
globals()["_last_panel_selection_signature"] = curr_selection_sig

matched_count = sum(1 for idx in selected_indices if _has_dataset_gt_for_panel_index(idx))
selected_class_counts = defaultdict(int)
for idx in selected_indices:
    selected_class_counts[int(test_patches[idx][1])] += 1
selected_dist = ", ".join(
    f"{CLASS_NAMES[lbl]}={selected_class_counts.get(lbl, 0)}" for lbl in range(len(CLASS_NAMES))
)
print(
    f"Panel selection: {matched_count}/{len(selected_indices)} samples have usable GT "
    f"(dataset mask or expected-empty no-tumor)."
)
print(f"Selected panel class counts: {selected_dist}")

for panel_i, sample_idx in enumerate(selected_indices):
    img_path, true_label = test_patches[sample_idx]

    # Build dual classifier input and retrieve both views.
    input_tensor, patch_pil, original_pil = _build_dual_input_tensor_from_patch(
        img_path,
        true_label,
        test_original_lookup,
        patch_transform,
        original_transform,
    )
    original_arr = np.array(original_pil.convert("RGB"))
    original_gray = np.array(original_pil.convert("L"))

    # Classification
    with torch.no_grad():
        logits = model(input_tensor.to(DEVICE))
        probs = torch.softmax(logits, dim=1)
        pred_class = probs.argmax(1).item()
        confidence = probs[0, pred_class].item()

    # Grad-CAM (overlay on raw MRI, not cropped patch)
    heatmap = generate_gradcam(model, input_tensor, target_layer, DEVICE, pred_class)
    heatmap = np.asarray(heatmap, dtype=np.float32).squeeze()
    if heatmap.shape != (INPUT_SIZE, INPUT_SIZE):
        if cv2 is not None:
            heatmap = cv2.resize(heatmap, (INPUT_SIZE, INPUT_SIZE), interpolation=cv2.INTER_LINEAR)
        else:
            heatmap = np.array(
                Image.fromarray((np.clip(heatmap, 0.0, 1.0) * 255).astype(np.uint8)).resize(
                    (INPUT_SIZE, INPUT_SIZE), Image.BILINEAR
                ),
                dtype=np.float32,
            ) / 255.0
    gradcam_overlay = overlay_gradcam(original_arr, heatmap)

    pred_class_name = CLASS_NAMES[int(pred_class)]

    # Classification-first panel logic:
    # only run segmentation when classifier predicts a tumor class.
    if pred_class_name == "notumor":
        pred_mask = np.zeros_like(original_gray, dtype=np.uint8)
        pred_mask_mode = "empty_for_notumor_prediction"
        pred_panel_title = "Predicted Mask (empty, no-tumor pred)"
    else:
        pred_mask_raw = predict_mask(seg_panel_model, original_gray, DEVICE, threshold=0.35)
        pred_mask = _refine_panel_mask(pred_mask_raw)
        pred_mask_mode = "segmentation_model"
        pred_panel_title = "Predicted Mask"

    # Ground-truth tumor mask: dataset mask only (no Otsu whole-brain fallback)
    cls_name = CLASS_NAMES[int(true_label)]
    lookup_key = (int(true_label), Path(img_path).name)
    original_img_path = str(test_original_lookup.get(lookup_key, img_path))

    gt_mask = np.zeros_like(original_gray, dtype=np.uint8)
    gt_mask_path = ""

    if cls_name == "notumor":
        # Classification "notumor" has no lesion annotation in segmentation dataset;
        # expected GT is an empty mask by design.
        gt_mask_mode = "no_tumor_empty"
        gt_panel_title = "Ground-Truth Mask (empty, no tumor)"
    else:
        gt_mask_path = _resolve_seg_mask_path(
            original_img_path,
            seg_gt_by_name,
            seg_gt_by_stem,
            seg_gt_by_norm,
            seg_gt_by_digits,
            expected_class=cls_name,
            original_gray=original_gray,
        )

        gt_mask_mode = "none"
        if gt_mask_path and Path(gt_mask_path).exists():
            gt_raw = np.array(Image.open(gt_mask_path).convert("L"))
            gt_bin = (gt_raw > 127).astype(np.uint8)
            gt_bin = _resize_binary_mask_to_shape(gt_bin, original_gray.shape[:2])
            gt_mask = _refine_panel_mask(gt_bin)
            if int(np.count_nonzero(gt_mask)) > 0:
                gt_mask_mode = "dataset"

        if gt_mask_mode == "dataset":
            gt_panel_title = "Ground-Truth Mask (dataset)"
        else:
            gt_panel_title = "Ground-Truth Mask (unavailable)"

    matched_seg_img_path = _matched_seg_image_path(gt_mask_path)
    print(
        f"[Panel {panel_i}] true={cls_name} | pred={pred_class_name} | patch={Path(img_path).name} | "
        f"original={Path(original_img_path).name} | "
        f"seg_image={Path(matched_seg_img_path).name if matched_seg_img_path else 'N/A'} | "
        f"gt_mask={Path(gt_mask_path).name if gt_mask_path else 'N/A'} | "
        f"gt_mode={gt_mask_mode} | pred_mask_mode={pred_mask_mode}"
    )
    print(f"  original_path: {original_img_path}")
    if matched_seg_img_path:
        print(f"  seg_image_path: {matched_seg_img_path}")
    if gt_mask_path:
        print(f"  gt_mask_path: {gt_mask_path}")

    # Stage consistency / report context: for predicted no-tumor, keep mask empty by design.
    if pred_class_name == "notumor":
        stage_iou = None
        report_segmentation_mask = None
        summary_segmentation_mask = None
    else:
        stage_iou = compute_stage_consistency_iou(pred_mask, heatmap)
        report_segmentation_mask = pred_mask
        summary_segmentation_mask = pred_mask

    # Generate report and show a concise summary in the right panel.
    report = vlm.generate_report(
        original_pil,
        pred_class,
        confidence,
        heatmap,
        segmentation_mask=report_segmentation_mask,
        stage_iou=stage_iou,
    )
    try:
        summary_text = summarize_report_text(
            report,
            predicted_class=pred_class,
            confidence=confidence,
            segmentation_mask=summary_segmentation_mask,
            true_label=true_label,
        )
    except TypeError:
        # Backward compatibility if an older summarize_report_text signature is still in memory.
        summary_text = summarize_report_text(report)

    # 5-panel plot (uncertainty map removed)
    panel_kwargs = {
        "original_img": original_arr,
        "context_img": original_arr,
        "gt_mask": gt_mask,
        "pred_mask": pred_mask,
        "gradcam_overlay": gradcam_overlay,
        "report_text": summary_text,
        "right_panel_title": "VLM Summary",
        "gt_panel_title": gt_panel_title,
        "pred_panel_title": pred_panel_title,
        "panel_linewidth": 2.8,
        "right_panel_width": 4.4,
        "text_wrap_width": 118,
        "include_uncertainty": False,
        "predicted_class": CLASS_NAMES[pred_class],
        "confidence": confidence,
        "save_path": str(OUTPUT_DIR / "figures" / f"qualitative_panel_{panel_i}.png"),
    }
    # Optional text styling args for newer plotting helper versions.
    plot_params = inspect.signature(plot_qualitative_panel).parameters
    if "gt_panel_title" not in plot_params:
        panel_kwargs.pop("gt_panel_title", None)
    if "pred_panel_title" not in plot_params:
        panel_kwargs.pop("pred_panel_title", None)
    if "report_fontsize" in plot_params:
        panel_kwargs["report_fontsize"] = 10.0
    if "header_fontsize" in plot_params:
        panel_kwargs["header_fontsize"] = 11.2

    plot_qualitative_panel(**panel_kwargs)

In [ ]:
# ============================================================
# Cell 4.2 - Clinical-style qualitative panel (no ground-truth mask)
# ============================================================
from pathlib import Path
import textwrap
import matplotlib.gridspec as gridspec
from matplotlib.patches import Rectangle
if "selected_indices" not in globals():
    raise RuntimeError("selected_indices not found. Run Cell 4.1 first, then run this cell.")
if "test_original_lookup" not in globals():
    test_original_lookup = _build_original_lookup(test_samples)
patch_transform = get_val_transforms(INPUT_SIZE)
original_transform = get_val_transforms(INPUT_SIZE)
seg_panel_model = globals().get("seg_panel_model", globals().get("seg_model", globals().get("resunet_model", None)))
if seg_panel_model is None or not hasattr(seg_panel_model, "eval"):
    raise RuntimeError("ResUNet model is not available. Run the segmentation train/load cell first.")
seg_panel_model.eval()
def _refine_panel_mask_local(mask: np.ndarray) -> np.ndarray:
    mask_bin = (np.asarray(mask) > 0).astype(np.uint8)
    if mask_bin.sum() == 0:
        return mask_bin
    if "cv2" not in globals() or cv2 is None:
        return mask_bin
    mask_bin = cv2.morphologyEx(mask_bin, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8), iterations=1)
    mask_bin = cv2.morphologyEx(mask_bin, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8), iterations=1)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_bin, connectivity=8)
    if num_labels > 1:
        largest_label = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
        mask_bin = (labels == largest_label).astype(np.uint8)
    mask_bin = cv2.dilate(mask_bin, np.ones((3, 3), np.uint8), iterations=1)
    return mask_bin.astype(np.uint8)
def _wrap_panel_text(text: str, width: int = 100) -> str:
    lines = []
    for line in (text or "").splitlines():
        s = line.strip()
        if not s:
            lines.append("")
            continue
        lines.extend(textwrap.wrap(s, width=width, break_long_words=False, break_on_hyphens=False) or [""])
    return "\n".join(lines)
def plot_qualitative_panel_no_gt(
    original_img: np.ndarray,
    pred_mask: np.ndarray,
    gradcam_overlay: np.ndarray,
    report_text: str,
    predicted_class: str,
    confidence: float,
    pred_panel_title: str = "Predicted Mask",
    right_panel_title: str = "VLM Summary",
    panel_linewidth: float = 2.8,
    right_panel_width: float = 4.4,
    text_wrap_width: int = 118,
    report_fontsize: float = 10.0,
    header_fontsize: float = 11.2,
    save_path: str = None,
) -> None:
    fig = plt.figure(figsize=(24, 6.4))
    gs = gridspec.GridSpec(1, 4, width_ratios=[1, 1, 1, right_panel_width])
    def _draw_border(ax):
        ax.add_patch(
            Rectangle(
                (0, 0), 1, 1,
                transform=ax.transAxes,
                fill=False,
                linewidth=panel_linewidth,
                edgecolor="#202020",
            )
        )
    with torch.no_grad():
        logits = model(input_tensor.to(DEVICE))
        probs = torch.softmax(logits, dim=1)
        pred_class = probs.argmax(1).item()
        confidence = probs[0, pred_class].item()
    pred_class_name = CLASS_NAMES[int(pred_class)]
    heatmap = generate_gradcam(model, input_tensor, target_layer, DEVICE, pred_class)
    heatmap = np.asarray(heatmap, dtype=np.float32).squeeze()
    if heatmap.shape != (INPUT_SIZE, INPUT_SIZE):
        if "cv2" in globals() and cv2 is not None:
            heatmap = cv2.resize(heatmap, (INPUT_SIZE, INPUT_SIZE), interpolation=cv2.INTER_LINEAR)
        else:
            heatmap = np.array(
                Image.fromarray((np.clip(heatmap, 0.0, 1.0) * 255).astype(np.uint8)).resize(
                    (INPUT_SIZE, INPUT_SIZE), Image.BILINEAR
                ),
                dtype=np.float32,
            ) / 255.0
    gradcam_overlay = overlay_gradcam(original_arr, heatmap)
    if pred_class_name == "notumor":
        pred_mask = np.zeros_like(original_gray, dtype=np.uint8)
        pred_panel_title = "Predicted Mask (empty, no-tumor pred)"
        stage_iou = None
        report_segmentation_mask = None
        summary_segmentation_mask = None
        pred_mask_mode = "empty_for_notumor_prediction"
    else:
        pred_mask_raw = predict_mask(seg_panel_model, original_gray, DEVICE, threshold=0.35)
        pred_mask = _refine_panel_mask_local(pred_mask_raw)
        pred_panel_title = "Predicted Mask"
        stage_iou = compute_stage_consistency_iou(pred_mask, heatmap)
        report_segmentation_mask = pred_mask
        summary_segmentation_mask = pred_mask
        pred_mask_mode = "segmentation_model"
    report = vlm.generate_report(
        original_pil,
        pred_class,
        confidence,
        heatmap,
        segmentation_mask=report_segmentation_mask,
        stage_iou=stage_iou,
    )
    try:
        summary_text = summarize_report_text(
            report,
            predicted_class=pred_class,
            confidence=confidence,
            segmentation_mask=summary_segmentation_mask,
            true_label=true_label,
        )
    except TypeError:
        summary_text = summarize_report_text(report)
    print(
        f"[No-GT Panel {panel_i}] true={CLASS_NAMES[int(true_label)]} | pred={pred_class_name} | "
        f"patch={Path(img_path).name} | pred_mask_mode={pred_mask_mode}"
    )
    plot_qualitative_panel_no_gt(
        original_img=original_arr,
        pred_mask=pred_mask,
        gradcam_overlay=gradcam_overlay,
        report_text=summary_text,
        predicted_class=pred_class_name,
        confidence=confidence,
        pred_panel_title=pred_panel_title,
        save_path=str(OUTPUT_DIR / "figures" / f"qualitative_panel_no_gt_{panel_i}.png"),
    )

In [ ]:
# ============================================================
# Cell 4.2b - Clinical-style qualitative panel (no ground-truth mask) [fixed]
# ============================================================
from pathlib import Path
import random
import textwrap

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Rectangle

try:
    import cv2
except ImportError:
    cv2 = None

if "target_layer" not in globals():
    target_layer = model.layer4[-1]

if "test_original_lookup" not in globals():
    test_original_lookup = _build_original_lookup(test_samples)

if "selected_indices" not in globals() or not selected_indices:
    if "test_patches" not in globals() or len(test_patches) == 0:
        raise RuntimeError("No selected_indices and test_patches is empty. Run earlier cells first.")
    _fallback = list(range(len(test_patches)))
    random.SystemRandom().shuffle(_fallback)
    selected_indices = _fallback[: min(8, len(_fallback))]
    print(f"selected_indices was empty; sampled {len(selected_indices)} fallback samples for Cell 4.2b.")

patch_transform = get_val_transforms(INPUT_SIZE)
original_transform = get_val_transforms(INPUT_SIZE)

seg_panel_model = globals().get("seg_panel_model", globals().get("seg_model", globals().get("resunet_model", None)))
if seg_panel_model is None or not hasattr(seg_panel_model, "eval"):
    raise RuntimeError("ResUNet model is not available. Run the segmentation train/load cell first.")
seg_panel_model.eval()


def _refine_panel_mask_local(mask: np.ndarray) -> np.ndarray:
    mask_bin = (np.asarray(mask) > 0).astype(np.uint8)
    if mask_bin.sum() == 0:
        return mask_bin
    if cv2 is None:
        return mask_bin
    mask_bin = cv2.morphologyEx(mask_bin, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8), iterations=1)
    mask_bin = cv2.morphologyEx(mask_bin, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8), iterations=1)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_bin, connectivity=8)
    if num_labels > 1:
        largest_label = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
        mask_bin = (labels == largest_label).astype(np.uint8)
    mask_bin = cv2.dilate(mask_bin, np.ones((3, 3), np.uint8), iterations=1)
    return mask_bin.astype(np.uint8)


def _wrap_panel_text(text: str, width: int = 100) -> str:
    lines = []
    for line in (text or "").splitlines():
        s = line.strip()
        if not s:
            lines.append("")
            continue
        lines.extend(textwrap.wrap(s, width=width, break_long_words=False, break_on_hyphens=False) or [""])
    return "\n".join(lines)


def plot_qualitative_panel_no_gt(
    original_img: np.ndarray,
    pred_mask: np.ndarray,
    gradcam_overlay: np.ndarray,
    report_text: str,
    predicted_class: str,
    confidence: float,
    pred_panel_title: str = "Predicted Mask",
    right_panel_title: str = "VLM Summary",
    panel_linewidth: float = 2.8,
    right_panel_width: float = 4.4,
    text_wrap_width: int = 118,
    report_fontsize: float = 10.0,
    header_fontsize: float = 11.2,
    save_path: str = None,
) -> None:
    fig = plt.figure(figsize=(24, 6.4))
    gs = gridspec.GridSpec(1, 4, width_ratios=[1, 1, 1, right_panel_width])

    def _draw_border(ax):
        ax.add_patch(
            Rectangle(
                (0, 0),
                1,
                1,
                transform=ax.transAxes,
                fill=False,
                linewidth=panel_linewidth,
                edgecolor="#202020",
            )
        )

    ax0 = fig.add_subplot(gs[0])
    ax0.imshow(original_img)
    ax0.set_title("Original MRI", fontsize=11)
    ax0.axis("off")
    _draw_border(ax0)

    ax1 = fig.add_subplot(gs[1])
    ax1.imshow(original_img)
    ax1.imshow((np.asarray(pred_mask) > 0).astype(np.float32), cmap="Blues", alpha=0.45)
    ax1.set_title(pred_panel_title, fontsize=11)
    ax1.axis("off")
    _draw_border(ax1)

    ax2 = fig.add_subplot(gs[2])
    ax2.imshow(gradcam_overlay)
    ax2.set_title("Grad-CAM", fontsize=11)
    ax2.axis("off")
    _draw_border(ax2)

    report_ax = fig.add_subplot(gs[3])
    report_ax.axis("off")
    _draw_border(report_ax)
    report_ax.set_title(right_panel_title, fontsize=11)

    header = f"Prediction: {predicted_class}  (conf: {confidence:.1%})"
    width_scale = max(1.0, float(right_panel_width) / 2.4)
    effective_wrap_width = max(40, min(160, int(text_wrap_width * width_scale)))
    wrapped = _wrap_panel_text(report_text, width=effective_wrap_width)

    report_ax.text(
        0.05,
        0.95,
        header,
        transform=report_ax.transAxes,
        fontsize=header_fontsize,
        fontweight="bold",
        verticalalignment="top",
    )
    report_ax.text(
        0.04,
        0.84,
        wrapped,
        transform=report_ax.transAxes,
        fontsize=report_fontsize,
        verticalalignment="top",
        wrap=True,
        linespacing=1.48,
        family="serif",
    )

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


(OUTPUT_DIR / "figures").mkdir(parents=True, exist_ok=True)

for panel_i, sample_idx in enumerate(selected_indices):
    img_path, true_label = test_patches[sample_idx]

    input_tensor, patch_pil, original_pil = _build_dual_input_tensor_from_patch(
        img_path,
        true_label,
        test_original_lookup,
        patch_transform,
        original_transform,
    )
    original_arr = np.array(original_pil.convert("RGB"))
    original_gray = np.array(original_pil.convert("L"))

    with torch.no_grad():
        logits = model(input_tensor.to(DEVICE))
        probs = torch.softmax(logits, dim=1)
        pred_class = probs.argmax(1).item()
        confidence = probs[0, pred_class].item()

    pred_class_name = CLASS_NAMES[int(pred_class)]

    heatmap = generate_gradcam(model, input_tensor, target_layer, DEVICE, pred_class)
    heatmap = np.asarray(heatmap, dtype=np.float32).squeeze()
    if heatmap.shape != (INPUT_SIZE, INPUT_SIZE):
        if cv2 is not None:
            heatmap = cv2.resize(heatmap, (INPUT_SIZE, INPUT_SIZE), interpolation=cv2.INTER_LINEAR)
        else:
            heatmap = np.array(
                Image.fromarray((np.clip(heatmap, 0.0, 1.0) * 255).astype(np.uint8)).resize(
                    (INPUT_SIZE, INPUT_SIZE), Image.BILINEAR
                ),
                dtype=np.float32,
            ) / 255.0
    gradcam_overlay = overlay_gradcam(original_arr, heatmap)

    if pred_class_name == "notumor":
        pred_mask = np.zeros_like(original_gray, dtype=np.uint8)
        pred_panel_title = "Predicted Mask (empty, no-tumor pred)"
        stage_iou = None
        report_segmentation_mask = None
        summary_segmentation_mask = None
        pred_mask_mode = "empty_for_notumor_prediction"
    else:
        pred_mask_raw = predict_mask(seg_panel_model, original_gray, DEVICE, threshold=0.35)
        pred_mask = _refine_panel_mask_local(pred_mask_raw)
        pred_panel_title = "Predicted Mask"
        stage_iou = compute_stage_consistency_iou(pred_mask, heatmap)
        report_segmentation_mask = pred_mask
        summary_segmentation_mask = pred_mask
        pred_mask_mode = "segmentation_model"

    report = vlm.generate_report(
        original_pil,
        pred_class,
        confidence,
        heatmap,
        segmentation_mask=report_segmentation_mask,
        stage_iou=stage_iou,
    )
    try:
        summary_text = summarize_report_text(
            report,
            predicted_class=pred_class,
            confidence=confidence,
            segmentation_mask=summary_segmentation_mask,
            true_label=true_label,
        )
    except TypeError:
        summary_text = summarize_report_text(report)

    print(
        f"[No-GT Panel {panel_i}] true={CLASS_NAMES[int(true_label)]} | pred={pred_class_name} | "
        f"patch={Path(img_path).name} | pred_mask_mode={pred_mask_mode}"
    )

    plot_qualitative_panel_no_gt(
        original_img=original_arr,
        pred_mask=pred_mask,
        gradcam_overlay=gradcam_overlay,
        report_text=summary_text,
        predicted_class=pred_class_name,
        confidence=confidence,
        pred_panel_title=pred_panel_title,
        save_path=str(OUTPUT_DIR / "figures" / f"qualitative_panel_no_gt_{panel_i}.png"),
    )
